# Chapter 01-06 · Charts that say something, and randomness you can repeat

**Label:** Optional  |  **Time:** ~45 minutes  |  **Difficulty:** gentle

**Prerequisites:** 01-05, or the equivalent fluency with pandas.

**Position in the learning path:** module 01, chapter 6 of 6 - the last of the optional bridge.
After this: **02-01**, where the course proper resumes.

---

## Why this matters

Two topics, one idea: **a result you cannot show honestly and cannot repeat is not a result.**

A chart is an argument. The same twelve numbers can be drawn to say "we are growing fast" or "we
are flat", and both charts are truthful in the sense that no number was altered. Being able to
build the misleading one deliberately is the only reliable way to stop building it by accident.

A random seed is a promise. Without one, "model B is better" can be an artefact of which rows
happened to land in the test set - and in the demonstration below, the winner changes depending on
nothing but the split, while the *difference between splits* is ten times larger than the
difference between the models.

## What you will be able to do

By the end of this chapter you can:

1. **Build** a labelled chart with units, and choose the right chart type for a question.
2. **Construct** a misleading chart deliberately, and name the two techniques that did it.
3. **Use** `np.random.default_rng(seed)` and explain why it is safer than `np.random.seed`.
4. **Demonstrate** that an unseeded comparison can reverse its conclusion, and quantify how much
   of a score is split-to-split noise.
5. **State** what else must be fixed, beyond a seed, for a result to be reproducible.

## Warm-up: retrieve, do not reread

From memory:

1. What two lines make a join safe?
2. What is the difference between `agg` and `transform`?
3. Why does a missing day break a lag feature?
4. What does `errors="coerce"` do, and what must you always do after it?

<br>

*Answers: (1) compare the row count before and after, and pass `validate=`. (2) `agg` collapses to
one row per group; `transform` broadcasts the group value back onto every original row. (3) "the
previous row" stops meaning "the previous day", so every lag after the gap is off. (4) turns
unconvertible values into `NaN` - always count how many.*

## The situation

Maria asks a simple question: **"is the stand getting busier?"**

You have twelve months of rentals. You will answer it twice, with two charts, and get two
different answers.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

months = pd.date_range("2023-01-01", periods=12, freq="MS")
rentals = np.array([1040, 1010, 950, 1005, 970, 1020, 960, 995, 940, 980, 1030, 1060])
monthly = pd.Series(rentals, index=months, name="rentals")

print(f"first month {rentals[0]}, last month {rentals[-1]}  ->  {(rentals[-1] / rentals[0] - 1):+.1%} over the year")
print(f"first half mean {rentals[:6].mean():.0f}, second half mean {rentals[6:].mean():.0f}")

Over the full year, rentals went up by 1.9%, and the second half averaged slightly *below* the
first. The honest answer to Maria's question is **"no, it is flat"**.

Now here is the chart that says otherwise.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 4))

# The persuasive chart: last four months only, y-axis starting at 900.
recent = monthly.iloc[-4:]
left.plot(recent.index, recent.values, marker="o", color="#009E73", linewidth=2.5)
left.set_ylim(900, 1080)
left.set_title("Rentals are up 13%")
left.set_ylabel("rentals")

# The honest chart: every month, y-axis from zero.
right.plot(monthly.index, monthly.values, marker="o", color="#0072B2", linewidth=1.8)
right.axhline(monthly.mean(), color="grey", linestyle="--", linewidth=1, label=f"mean {monthly.mean():.0f}")
right.set_ylim(0, 1200)
right.set_title("Monthly rentals, 2023 (all 12 months)")
right.set_ylabel("Bikes rented per month (count)")
right.set_xlabel("Month")
right.legend()

fig.tight_layout()
plt.show()

## Failure lab: two charts, one dataset, opposite conclusions

The left-hand chart is not a lie. Every point on it is a real measured value, correctly plotted.
It is also completely misleading, and it does its work with exactly two techniques.

**1. A truncated y-axis.** Starting at 900 rather than 0 magnifies a 120-unit change into most of
the plot's height. The eye reads *the height of the line*, and the height of the line is now a
statement about the axis, not about the data.

**2. A chosen window.** Four months, picked because they happen to rise. Eight months of data
exist and are not shown. Nothing about the chart reveals that a choice was made.

Neither technique alters a number. Both alter the conclusion.

### When is a truncated axis legitimate?

Often. If body temperature is plotted from 0 °C, a fever is invisible - and the range that matters
is 36 to 40. The rule is not "always start at zero"; it is:

> **Start at zero when the length of the bar or the height of the line is the message. Truncate
> only when the variation is the message, and say so on the chart.**

Bar charts almost always need zero, because a bar's meaning *is* its length. Line charts of a
quantity that never goes near zero often should be truncated, with a label that makes the range
obvious. The dishonest version is the one that truncates silently to make a small change look big.

### The checklist for a chart you are about to show someone

| Ask | Because |
|---|---|
| Do both axes have a label **and a unit**? | "Rentals" is not a unit. "Bikes rented per month" is |
| Does the y-axis start at zero, and if not, is that visible? | The single most common way to mislead |
| Is the full available range shown, or did I choose a window? | If you chose, say why on the chart |
| Would a colourblind reader see the same distinctions? | 1 man in 12 cannot separate red from green |
| Does the title state the finding, or the topic? | A title that states a finding must be one the chart supports |
| Would the opposite conclusion be visible if it were true? | If not, the chart cannot inform anyone |

## Four chart types, four questions

Choose by the question, not by taste. Nearly everything in this course uses one of these.

| Question | Chart | Watch out for |
|---|---|---|
| How does this change over time? | **Line** | Gaps in the index - a line drawn straight across a missing day implies data that is not there (01-05) |
| How do two quantities relate? | **Scatter** | Overplotting when there are many points - use transparency or fewer points |
| How is one quantity distributed? | **Histogram** | The bin count changes the story; try several before believing one |
| How do categories compare? | **Bar** | Start at zero. Sort by value, not alphabetically, unless the order means something |

In [ ]:
rng = np.random.default_rng(11)
temperature = rng.normal(18, 6, 400)
demand = 40 + 2.5 * temperature + rng.normal(0, 12, 400)     # SYNTHETIC

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))

axes[0].scatter(temperature, demand, s=10, alpha=0.35, color="#0072B2")
axes[0].set_xlabel("Temperature (°C)"); axes[0].set_ylabel("Rentals per day (count)")
axes[0].set_title("Relationship")

axes[1].hist(demand, bins=25, color="#0072B2", edgecolor="white")
axes[1].set_xlabel("Rentals per day (count)"); axes[1].set_ylabel("Number of days")
axes[1].set_title("Distribution")

by_site = pd.Series({"north": 915, "south": 210, "east": 604})
by_site.sort_values(ascending=False).plot.bar(ax=axes[2], color="#0072B2", rot=0)
axes[2].set_ylim(0, None); axes[2].set_ylabel("Total rentals (count)")
axes[2].set_title("Comparison")

fig.tight_layout()
plt.show()

Three deliberate choices in that code, each of which is a habit rather than a trick:

- **`alpha=0.35` on the scatter.** With 400 points, solid markers pile up and dense regions look
  identical to sparse ones. Transparency restores the density, which is usually the interesting
  part.
- **`set_ylim(0, None)` on the bar chart.** Bars mean length; length must start at zero.
- **`sort_values(ascending=False)`.** Alphabetical order is an accident of naming. Sorted order is
  the finding.

**On colour.** Roughly one man in twelve cannot reliably distinguish red from green, so a
red-versus-green comparison is unreadable for a substantial minority. The palette used throughout
this course - blue `#0072B2`, orange `#D55E00`, green `#009E73`, pink `#CC79A7` - is a
colourblind-safe set. Better still, **do not let colour be the only signal**: pair it with a marker
shape, a line style, or a direct label, as chapter 00-01's first scatter did.

---

## Randomness you can repeat

Machine learning is full of randomness: which rows go into the test set, how a forest picks its
features, where a neural network starts. Left unseeded, every run gives a different answer - and
"different" is fine right up until it changes the conclusion.

### Predict before running

We compare two models on 60 rows. They are genuinely very close in quality.

1. Over 200 different random splits, how often will plain linear regression beat ridge?
2. Which will vary more - the difference *between the two models*, or the difference *between
   splits*?

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)
n = 60
X = rng.normal(0, 1, (n, 3))
y = 2 * X[:, 0] - X[:, 1] + rng.normal(0, 2, n)         # SYNTHETIC

for seed in (3, 6):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35, random_state=seed)
    linear = mean_absolute_error(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))
    ridge = mean_absolute_error(y_te, Ridge(alpha=3.0).fit(X_tr, y_tr).predict(X_te))
    print(f"random_state={seed}: linear {linear:.3f}, ridge {ridge:.3f}  ->  "
          f"{'ridge' if ridge < linear else 'linear'} wins")

Two seeds, two opposite conclusions, identical code and identical data. If you had run this once
and reported it, you would have reported whichever answer your machine happened to produce.

Now run it 200 times.

In [ ]:
differences, linear_wins = [], 0
for _ in range(200):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35)      # deliberately unseeded
    linear = mean_absolute_error(y_te, LinearRegression().fit(X_tr, y_tr).predict(X_te))
    ridge = mean_absolute_error(y_te, Ridge(alpha=3.0).fit(X_tr, y_tr).predict(X_te))
    differences.append(linear - ridge)
    linear_wins += linear < ridge

differences = np.array(differences)
print(f"linear wins {linear_wins} of 200 splits, ridge wins {200 - linear_wins}")
print(f"difference between the MODELS : mean {differences.mean():+.3f}")
print(f"difference between the SPLITS : {differences.max() - differences.min():.3f} "
      f"(best split to worst)")
print(f"the split matters {(differences.max() - differences.min()) / abs(differences.mean()):.0f}x "
      f"more than the model choice")

### Diagnosis

**Ridge wins most of the splits and linear wins a large minority.** So ridge probably is slightly
better here - but a single run reports the winner with a great deal of noise attached, and "ridge
beat linear" from one split is close to meaningless.

That cell is deliberately unseeded, so your exact counts will differ from the ones printed when
this notebook was written. That is the point rather than a flaw: run it twice and watch the numbers
move.

**Now look at the last line of the output.** The average difference between the two models is a
hundredth of a bike or so. The difference between the best and worst split is tens of times larger.

That is the sentence to take away:

> **Which rows landed in the test set mattered ten times more than which model you chose.**

Reporting one split's number as "the" score is therefore reporting mostly noise. The fix is not a
better seed; it is to **measure the noise** - cross-validation, which averages over many splits and
lets you see the spread. That is 04-07 and 07-03, and this is the demonstration that makes them
necessary rather than ceremonial.

### Two ways to seed, and why one is safer

In [ ]:
# The modern way: an explicit generator object.
gen_a = np.random.default_rng(42)
gen_b = np.random.default_rng(42)
print("two generators, same seed:", gen_a.normal(size=3).round(3), gen_b.normal(size=3).round(3))

# The legacy way: one hidden global generator.
np.random.seed(42)
first = np.random.normal(size=3)
_ = np.random.normal(size=100)          # some other code, somewhere, draws numbers
second = np.random.normal(size=3)
print("\nglobal seed, before and after an unrelated draw:")
print(" ", first.round(3))
print(" ", second.round(3), "  <- different, and nothing in your code changed")

`np.random.seed` sets **one hidden generator shared by the whole process**. Anything that draws
from it - a library, a helper function, a cell you ran earlier - advances it. Your results then
depend on what else ran first, which is why a notebook can be reproducible when run top to bottom
and irreproducible when a cell is re-run.

`np.random.default_rng(seed)` gives you **your own generator**, which nothing else can touch. Pass
it into functions that need randomness rather than reaching for a global. Every notebook in this
course does this, which is why you have seen `rng = np.random.default_rng(...)` in every chapter.

**scikit-learn's version of the same idea is `random_state`.** Every estimator and splitter that
uses randomness accepts it, and setting it is not optional in work anyone else will read.

### A seed is necessary and not sufficient

Setting a seed fixes the random draws. Reproducing a *result* needs more, and 04-08 builds the
full checklist. The short version:

| Also pin | Because |
|---|---|
| Library versions | A default changed between scikit-learn releases and your number moves |
| The data, exactly | "The customers table" is different today than last Tuesday |
| The order of operations | Shuffling then splitting is not the same as splitting then shuffling |
| Every `random_state` | One unseeded splitter is enough to make the whole run irreproducible |

And one thing a seed cannot fix: **a seeded result is repeatable, not correct.** Fixing
`random_state=42` makes your number stable; it does not make it representative. If the number
changes a lot with the seed - as it does above - the honest report is a range from many splits, not
one number from a lucky one.

**The anti-pattern to name out loud:** trying several seeds and keeping the one that gives the best
score. That is choosing your test set to flatter your model, and it is 07-04's subject.

## Common misconceptions

**"Charts are for the end, after the analysis."**
Charts are how you find things. A scatter plot takes ten seconds and shows a cluster, a ceiling, a
gap, a group of impossible values - things no summary statistic will report. Module 02 is largely
this argument.

**"Every bar chart must start at zero, and so must every line chart."**
Bars, yes - length is the message. Lines of a quantity that never approaches zero, usually not.
The rule is to make the range visible, not to obey a slogan.

**"A trend line makes the relationship real."**
`plt` will fit a line through anything, including noise. A line drawn over a cloud with no
relationship looks exactly as convincing as one drawn over a real effect. 03-03 gives you the tool
for saying which you have.

**"`random_state=42` makes my result correct."**
It makes it repeatable. If the result changes a lot with the seed, repeatability is precisely what
lets you discover that - by trying more seeds, not by keeping the first.

**"Reproducible means someone else gets my number."**
It means someone else gets your number *on their machine, next year, from your description*. That
needs versions, data and order pinned as well as seeds.

**"Setting `np.random.seed(0)` at the top of the notebook is enough."**
It is enough only if nothing else draws from the global generator and no cell is ever re-run out of
order. Both assumptions fail in practice, which is why `default_rng` exists.

---

## Exercises

Solutions: `solutions/01_python_bridge/01-06_plotting_and_seeds_solutions.ipynb`.

### Quick understanding

**E1 (define).** Name the two techniques that made the misleading chart, and say what each one
does to the reader.

**E2 (explain).** When is a truncated y-axis legitimate? Give one example where it is required and
one where it is dishonest.

**E3 (explain).** Why is `np.random.default_rng(0)` safer than `np.random.seed(0)`?

### Hand calculation

**E4 (calculate).** Monthly rentals: 1040, 1010, 950, 1005, 970, 1020, 960, 995, 940, 980, 1030,
1060. Compute: the change from first to last month as a percentage; the change from the *ninth* to
the last month as a percentage; and the difference between the first-half and second-half means.
Which of the three would you put in a headline, and what would you have to say alongside it?

**E5 (calculate).** A model scores MAE 1.47 on one split. Across twenty splits the scores range
from 1.24 to 1.92. Write one sentence reporting this result honestly, and one sentence reporting
it dishonestly without stating anything false.

### Coding

**E6 (code).** Redraw the misleading chart honestly: all twelve months, a zero-based axis, labelled
units, and a title that states what the data supports. Then add a second version that truncates the
axis *legitimately* - showing the variation clearly while making the truncation obvious.

**E7 (code).** Write `compare_models(X, y, n_splits=200)` returning, for each of two models, the
mean, standard deviation, minimum and maximum MAE across `n_splits` random splits - plus the
proportion of splits each one wins. Run it and write the one-sentence conclusion you would put in a
report.

### Interpretation

**E8 (interpret).** A colleague shows a histogram of daily rentals with 5 bins and concludes the
distribution is "roughly normal". What would you check before agreeing, and what could 5 bins be
hiding?

### Debugging

**E9 (diagnose).** A notebook produces the same result every time it is run top to bottom, but a
different result when the modelling cell is re-run on its own. There is `np.random.seed(0)` at the
top. Explain, and give the fix.

### Exam and interview reasoning

**E10 (defend).** *"We tried five random seeds and reported the best one - it's still a real
result, we didn't change any data."* Reply in four sentences.

**E11 (design).** You must present one chart to Maria answering "is the stand getting busier?".
Describe the chart you would draw - type, axes, range, what else is on it - and the one sentence
you would say while showing it.

### Transfer to a different situation

**E12 (design).** A hospital dashboard shows "average waiting time this week: 24 minutes" as a
single large number. Describe two charts that would tell the story better, and say what each one
would reveal that the single number cannot.

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to Maria why the first chart was not a lie and was
still not to be trusted.

### Optional challenge

**E14 (code + diagnose).** Quantify how much of a reported score is noise. For the chapter's data,
compute the distribution of ridge's MAE over 500 splits, then draw a histogram with the single-split
value from `random_state=3` marked on it. Compute what percentile that single value sits at, and
write the sentence you would use to warn a colleague against reporting one split.

In [ ]:
# Your workspace. Still in memory: monthly, rentals, X, y, temperature, demand, differences.

## Mastery check

Without scrolling up, can you:

- [ ] Name the two techniques that made the misleading chart? *(If not: "Failure lab".)*
- [ ] State the rule for when to start an axis at zero? *(If not: "When is a truncated axis
      legitimate?".)*
- [ ] Choose between line, scatter, histogram and bar from a question? *(If not: "Four chart
      types".)*
- [ ] Say why `default_rng` is safer than the global seed? *(If not: "Two ways to seed".)*
- [ ] Say what a seed does **not** guarantee? *(If not: "Necessary and not sufficient".)*

## What should now feel instinctive

1. **Label both axes, with units.** Every time, including throwaway charts - especially those,
   because they are the ones that end up in a slide.
2. **Ask what was left off the chart** - which months, which rows, which axis range.
3. **Make randomness explicit**: an `rng` you created, and a `random_state` on everything that
   takes one.
4. **One split is one sample.** If a difference is small, measure the spread before believing it.
5. **A chart is an argument.** Draw the one that would let someone disagree with you if they were
   right.

## Flashcards

| Question | Answer |
|---|---|
| The two ways to mislead with a truthful chart | Truncate the axis; choose the window |
| When must an axis start at zero? | Whenever length is the message - bar charts almost always |
| Line, scatter, histogram, bar - which question each? | Change over time; relationship; distribution; comparison of categories |
| Why transparency on a scatter? | Overlapping points hide density, which is usually the interesting part |
| Colourblind-safe habit | Never let colour be the only signal - add shape, style or a direct label |
| `default_rng` vs `np.random.seed` | Your own generator vs one hidden global one that anything can advance |
| scikit-learn's equivalent | `random_state=` on every estimator and splitter that uses randomness |
| What does a seed guarantee? | Repeatability, not correctness and not representativeness |
| What else must be pinned? | Library versions, the data, the order of operations, every random_state |
| Why is one split's score mostly noise here? | The spread across splits was ten times the difference between the models |

## Next

**Module 02 · Data literacy and EDA**, starting with **02-01 · What is a row?**

Module 01 is finished. You can load a file and check it, select and filter without surprises,
group, join and index by time, draw a chart that is honest, and make a run repeat. That is the
whole toolkit the rest of the course uses.

Module 02 turns those tools on the harder question: not "how do I compute this?" but "what does
this data actually mean, and what is wrong with it?" - which is where the expensive mistakes live.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).